# Week 3
# Data Cleaning & Dataset Preparation

This notebook performs data cleaning based on the findings from Week 2.

Objectives:

- Remove duplicate records
- Handle invalid values
- Remove unnecessary columns
- Keep only business-relevant fields
- Export clean datasets for future analysis

## Part 1: Load Validated Dataset

Load the residential listing and sold datasets generated from Week 1.

These datasets were validated during Week 2 and will now be cleaned for future analysis.

In [58]:
import pandas as pd
import numpy as np

listings = pd.read_csv(
    "../outputs/combined_listings_residential.csv",
    low_memory=False
)

sold = pd.read_csv(
    "../outputs/combined_sold_residential.csv",
    low_memory=False
)

print(listings.shape)
print(sold.shape)

(616048, 84)
(450699, 82)


## Part 2: Remove Duplicate Columns

Some listing files contain duplicated columns with a `.1` suffix.

These duplicated fields are removed to ensure a consistent schema.

In [59]:
duplicate_cols = [
    col for col in listings.columns
    if col.endswith(".1")
]

listings = listings.drop(columns=duplicate_cols)

print("Listings:", listings.shape)

Listings: (616048, 73)


## Part 3: Remove High-Missing Columns

According to the updated project guideline, columns with more than 90% missing values can be removed because they are unlikely to contribute meaningful information to the final Market Analysis and Competitive Analysis dashboards.

In [60]:
listing_missing = listings.isnull().mean() * 100
sold_missing = sold.isnull().mean() * 100

listing_drop_cols = listing_missing[
    listing_missing > 90
].index

sold_drop_cols = sold_missing[
    sold_missing > 90
].index

listings = listings.drop(columns=listing_drop_cols)
sold = sold.drop(columns=sold_drop_cols)

print("Listings:", listings.shape)
print("Sold:", sold.shape)

Listings: (616048, 60)
Sold: (450699, 67)



Columns with more than 90% missing values were removed to improve data quality while preserving fields that are useful for the final dashboards.

## Part 4: Remove Duplicate Records

Duplicate rows were identified during Week 2.

They are now removed to eliminate redundant transactions.

In [61]:
listing_before = len(listings)
sold_before = len(sold)

listings = listings.drop_duplicates()
sold = sold.drop_duplicates()

print(f"Listings: {listing_before} → {len(listings)}")
print(f"Sold: {sold_before} → {len(sold)}")

Listings: 616048 → 616048
Sold: 450699 → 439206


## Part 5: Handle Invalid Numeric Values

Several numeric fields contain invalid values such as zero prices, negative days on market, and zero living areas.

These values are converted to missing values for future analysis.

In [62]:
sold.loc[
    sold["ClosePrice"] <= 0,
    "ClosePrice"
] = np.nan

sold.loc[
    sold["OriginalListPrice"] <= 0,
    "OriginalListPrice"
] = np.nan

sold.loc[
    sold["LivingArea"] <= 0,
    "LivingArea"
] = np.nan

sold.loc[
    sold["DaysOnMarket"] < 0,
    "DaysOnMarket"
] = np.nan

In [63]:
print(sold[[
    "ClosePrice",
    "OriginalListPrice",
    "LivingArea",
    "DaysOnMarket"
]].isnull().sum())

ClosePrice             3
OriginalListPrice    833
LivingArea           403
DaysOnMarket          46
dtype: int64


## Part 6: Review Extreme Outliers

Extreme values were reviewed using the 99th percentile.

Outliers are retained for now because luxury properties may represent valid observations.

In [64]:
for col in [
    "ClosePrice",
    "LivingArea",
    "DaysOnMarket"
]:

    threshold = sold[col].quantile(.99)

    print(col)

    print(
        sold[
            sold[col] > threshold
        ][col].describe()
    )

ClosePrice
count    4.351000e+03
mean     1.463757e+07
std      5.944148e+07
min      5.602000e+06
25%      6.400000e+06
50%      7.600000e+06
75%      1.030000e+07
max      9.895000e+08
Name: ClosePrice, dtype: float64
LivingArea
count    4.389000e+03
mean     1.097705e+04
std      2.568408e+05
min      5.291000e+03
25%      5.680000e+03
50%      6.275000e+03
75%      7.540000e+03
max      1.702132e+07
Name: LivingArea, dtype: float64
DaysOnMarket
count     4379.000000
mean       323.825074
std        214.511057
min        234.000000
25%        255.500000
50%        287.000000
75%        343.000000
max      12430.000000
Name: DaysOnMarket, dtype: float64


## Part 7: Feature  Inventory

Before cleaning the dataset, all available fields are reviewed.

For each column, we summarize:

- Data type
- Missing percentage
- Business meaning
- Whether it supports the final dashboard
- Keep / Drop decision

In [65]:
feature_summary = pd.DataFrame({
    "Column": sold.columns,
    "Data Type": sold.dtypes.astype(str).values,
    "Missing %": (sold.isnull().mean()*100).round(2).values,
    "Missing Count": sold.isnull().sum().values,
    "Unique Values": sold.nunique().values,
    "Example Value": sold.iloc[0].values
})

feature_summary

,Column,Data Type,Missing %,Missing Count,Unique Values,Example Value
0,BuyerAgentAOR,object,10.43,45811,63,Mlslistings
1,ListAgentAOR,object,8.53,37478,61,Mlslistings
2,Flooring,object,35.75,157037,334,"Carpet,Tile,Wood"
3,ViewYN,object,8.78,38545,2,True
4,PoolPrivateYN,object,8.76,38467,2,False
...,...,...,...,...,...,...
62,LotSizeSquareFeet,float64,7.71,33850,42034,NaN
63,OriginatingSystemName,object,79.59,349579,1,CRMLS
64,OriginatingSystemSubName,object,79.59,349579,10,CRMLS_MLSL
65,BuyerAgencyCompensationType,object,91.48,401792,3,NaN


In [66]:
feature_summary.to_csv(
    "../outputs/feature_dictionary.csv",
    index=False
)

In [67]:
feature_dictionary = pd.read_excel(
    "/Users/wing/IDX intern/outputs/feature_dictionary_filled.xlsx"
)

feature_dictionary.head()

drop_columns = feature_dictionary.loc[
    feature_dictionary["Decision"] == "Drop",
    "Column"
].tolist()

print(drop_columns)

['OriginatingSystemName', 'OriginatingSystemSubName', 'BuyerAgencyCompensationType', 'BuyerAgencyCompensation']


In [68]:
listings = listings.drop(
    columns=[c for c in drop_columns if c in listings.columns],
    errors="ignore"
)

sold = sold.drop(
    columns=[c for c in drop_columns if c in sold.columns],
    errors="ignore"
)

print("Listings:", listings.shape)
print("Sold:", sold.shape)

Listings: (616048, 58)
Sold: (439206, 63)


### Results

The following columns were removed from the working datasets:

- **OriginatingSystemName**
- **OriginatingSystemSubName**
- **BuyerAgencyCompensationType**
- **BuyerAgencyCompensation**

These fields were excluded because they are either:

- System metadata that does not contribute to business analysis, or
- Variables with more than **90% missing values**, making them unsuitable for the final Market Analysis and Competitive Analysis dashboards.

The dataset is now cleaner and contains only features that are more relevant for downstream analysis and dashboard development.

## Part 8: Export Clean Dataset

The cleaned datasets are exported for subsequent market analysis and dashboard development.

In [69]:
listings.to_csv(
    "../outputs/clean_listings.csv",
    index=False
)

sold.to_csv(
    "../outputs/clean_sold.csv",
    index=False
)

print("Clean datasets exported.")

Clean datasets exported.


# Part 9: Mortgage Rate Enrichment

To support downstream market analysis, the cleaned MLS datasets are enriched with the national 30-year fixed mortgage rate published by the Federal Reserve (FRED).

Because mortgage rates are reported weekly while MLS transactions are analyzed monthly, the weekly observations are first aggregated into monthly averages before merging.

In [73]:
import pandas as pd

url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"

mortgage = pd.read_csv(url)

print(mortgage.head())
print(mortgage.columns)

  observation_date  MORTGAGE30US
0       1971-04-02          7.33
1       1971-04-09          7.31
2       1971-04-16          7.31
3       1971-04-23          7.31
4       1971-04-30          7.29
Index(['observation_date', 'MORTGAGE30US'], dtype='object')


In [74]:
import pandas as pd

url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"

mortgage = pd.read_csv(
    url,
    parse_dates=["observation_date"]
)

# Rename columns
mortgage.columns = [
    "date",
    "rate_30yr_fixed"
]

mortgage.head()

,date,rate_30yr_fixed
0,1971-04-02,7.33
1,1971-04-09,7.31
2,1971-04-16,7.31
3,1971-04-23,7.31
4,1971-04-30,7.29


In [76]:
mortgage["year_month"] = mortgage["date"].dt.to_period("M")

mortgage_monthly = (
    mortgage
    .groupby("year_month")["rate_30yr_fixed"]
    .mean()
    .reset_index()
)

mortgage_monthly.head()

,year_month,rate_30yr_fixed
0,1971-04,7.3100
1,1971-05,7.4250
2,1971-06,7.5300
3,1971-07,7.6040
4,1971-08,7.6975


In [77]:
sold["year_month"] = (
    pd.to_datetime(
        sold["CloseDate"]
    ).dt.to_period("M")
)

listings["year_month"] = (
    pd.to_datetime(
        listings["ListingContractDate"]
    ).dt.to_period("M")
)

In [78]:
sold = sold.merge(
    mortgage_monthly,
    on="year_month",
    how="left"
)

listings = listings.merge(
    mortgage_monthly,
    on="year_month",
    how="left"
)

In [79]:
print(
    "Missing mortgage rates (Sold):",
    sold["rate_30yr_fixed"].isnull().sum()
)

print(
    "Missing mortgage rates (Listings):",
    listings["rate_30yr_fixed"].isnull().sum()
)

Missing mortgage rates (Sold): 0
Missing mortgage rates (Listings): 0


In [81]:
sold[
    ["CloseDate","year_month","rate_30yr_fixed"]
].head()

listings[
    ["ListingContractDate","year_month","rate_30yr_fixed"]
].head()

,ListingContractDate,year_month,rate_30yr_fixed
0,2024-01-01,2024-01,6.6425
1,2024-01-24,2024-01,6.6425
2,2024-01-12,2024-01,6.6425
3,2024-01-20,2024-01,6.6425
4,2024-01-12,2024-01,6.6425


### Results

The mortgage rate was successfully merged into both datasets.

The new feature (`rate_30yr_fixed`) will be available for downstream market trend analysis and dashboard development.

In [82]:
sold.to_csv(
    "../outputs/clean_sold_with_rates.csv",
    index=False
)

listings.to_csv(
    "../outputs/clean_listings_with_rates.csv",
    index=False
)